In [1]:
import numpy as np, matplotlib.pyplot as plt, pandas as pd

pd.set_option("display.max_rows", 8)
!date

Fri 09 Aug 2024 03:29:24 PM PDT


# Mean deaths and stillbirths averted by adding folate by wealth quintile


In [2]:
import vivarium_inputs
import db_queries
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [3]:
location = "india"
vehicle = "rice"
intervention_scenario = "intervention"

In [4]:
# Parameters
location = "ethiopia"
vehicle = "salt"
intervention_scenario = "intervention_25_nrv"

## Forecasted births and stillbirths

In [5]:
asfr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.age_specific_fertility_rate,
    "estimate",
    location.title(),
    years=2022,
).value

In [6]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)

In [7]:
# Scale ASFR in each category down proportionally to the scale-down in TFR forecasted from GBD 2017
if location == "india":
    asfr_2030_to_2022_ratio = 1.61 / 1.91  # http://ihmeuw.org/6j8s
elif location == "nigeria":
    asfr_2030_to_2022_ratio = 4.43 / 4.96  # http://ihmeuw.org/6jqx
elif location == "ethiopia":
    asfr_2030_to_2022_ratio = 3.27 / 4.10  # http://ihmeuw.org/6j7d

asfr = asfr * asfr_2030_to_2022_ratio
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end
Ethiopia  Female  10.0       15.0     2022        2023        0.002040
                  15.0       20.0     2022        2023        0.036179
                  20.0       25.0     2022        2023        0.152703
                  25.0       30.0     2022        2023        0.146699
                                                                ...   
                  35.0       40.0     2022        2023        0.100878
                  40.0       45.0     2022        2023        0.052838
                  45.0       50.0     2022        2023        0.017421
                  50.0       55.0     2022        2023        0.001615
Name: value, Length: 9, dtype: float64

In [8]:
asfr = (
    asfr.reset_index()
    .assign(year_start=2030, year_end=2031)
    .set_index(asfr.index.names)
    .value
)
asfr.sort_values()

location  sex     age_start  age_end    year_start  year_end
Ethiopia  Female  0.000000   0.019178   2030        2031        0.000000
          Male    0.076712   0.500000   2030        2031        0.000000
                  0.500000   1.000000   2030        2031        0.000000
                  1.000000   2.000000   2030        2031        0.000000
                                                                  ...   
          Female  35.000000  40.000000  2030        2031        0.100878
                  30.000000  35.000000  2030        2031        0.142050
                  25.000000  30.000000  2030        2031        0.146699
                  20.000000  25.000000  2030        2031        0.152703
Name: value, Length: 50, dtype: float64

In [9]:
from vivarium_inputs import utilities
from vivarium_inputs.utility_data import get_location_id
from vivarium_gbd_access.gbd import get_age_group_id, SEX, RELEASE_IDS

In [10]:
def get_population_future(location, year):
    # Cobbled together from pieces of vivarium_inputs and vivarium_gbd_access
    # TODO: vivarium_inputs should be able to get forecasted pop!
    location_id = get_location_id(location)
    year_id = year
    data = db_queries.get_population(
        age_group_id=get_age_group_id(),
        location_id=location_id,
        year_id=year_id,
        sex_id=SEX.MALE + SEX.FEMALE + SEX.COMBINED,
        release_id=RELEASE_IDS.GBD_2021,
        forecasted_pop=True,
    )
    data = utilities.normalize_sex(
        data.drop("run_id", axis="columns").rename(columns={"population": "value"}),
        fill_value=None,
        cols_to_fill=utilities.DRAW_COLUMNS,
    )
    data = utilities.reshape(data, ["value"])
    data = utilities.scrub_gbd_conventions(data, location)
    data = utilities.split_interval(
        data, interval_column="age", split_column_prefix="age"
    )
    data = utilities.split_interval(
        data, interval_column="year", split_column_prefix="year"
    )
    return utilities.sort_hierarchical_data(data)

In [11]:
pop = get_population_future(location.title(), 2030).value.reindex(asfr.index)
pop

location  sex     age_start  age_end     year_start  year_end
Ethiopia  Female  0.000000   0.019178    2030        2031         34555.502815
                  0.019178   0.076712    2030        2031        102821.618998
                  0.076712   0.500000    2030        2031                  NaN
                  0.500000   1.000000    2030        2031                  NaN
                                                                     ...      
          Male    80.000000  85.000000   2030        2031        215000.475056
                  85.000000  90.000000   2030        2031        100478.541795
                  90.000000  95.000000   2030        2031         33156.550480
                  95.000000  125.000000  2030        2031          8245.199268
Name: value, Length: 50, dtype: float64

In [12]:
# Forecasted population does not have younger ages, but luckily none of these are WRA
assert (pop.index.get_level_values("age_end")[pop.isna()] < 10).all()
pop[pop.isna()]

location  sex     age_start  age_end  year_start  year_end
Ethiopia  Female  0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
          Male    0.076712   0.5      2030        2031       NaN
                  0.500000   1.0      2030        2031       NaN
                  1.000000   2.0      2030        2031       NaN
                  2.000000   5.0      2030        2031       NaN
Name: value, dtype: float64

In [13]:
pop = pop.fillna(0)

In [14]:
n_births = (pop * asfr).sum()
n_births

3717119.066255957

In [15]:
sbr = vivarium_inputs.get_measure(
    gbd_mapping.covariates.stillbirth_to_live_birth_ratio,
    "estimate",
    location.title(),
    years=2022,
).value
sbr

location  year_start  year_end  parameter  
Ethiopia  2022        2023      lower_value    0.017252
                                mean_value     0.017252
                                upper_value    0.017252
Name: value, dtype: float64

In [16]:
sbr = sbr[sbr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
sbr

location  year_start  year_end
Ethiopia  2022        2023        0.017252
Name: value, dtype: float64

In [17]:
sbr = sbr.values[0]

In [18]:
births_and_stillbirths = n_births + n_births * sbr
births_and_stillbirths / 1e6

3.7812475607071954

## Fertility (technically birth-and-stillbirth) disparities

In [19]:
if location == "india":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/IND/2019_2021/IND_DHS7_2019_2021_REP_FINAL_Y2022M05D11.PDF
        dict(  # Table 8.4 Perinatal mortality -- using "Number of pregnancies of 7 or more months' duration" as a proxy
            lowest=56_979,
            second=50_335,
            middle=45_189,
            fourth=42_611,
            highest=36_290,
        )
    )
elif location == "nigeria":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/NGA/2018/NGA_DHS7_2018_REP_QUEST_Y2019M11D05.PDF
        dict(  # Table 8.4 Perinatal mortality
            lowest=7_712,
            second=7_886,
            middle=7_139,
            fourth=6_328,
            highest=5_558,
        )
    )
elif location == "ethiopia":
    dist_births_and_stillbirths_by_wealth = pd.Series(  # from file:///J:/DATA/DHS_PROG_DHS/ETH/2016/ETH_DHS7_2016_REP_QUEST_Y2017M08D15.PDF
        dict(  # Table 8.4 Perinatal mortality
            lowest=2_645,
            second=2_516,
            middle=2_290,
            fourth=2_018,
            highest=1_592,
        )
    )

s_births = (
    n_births
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births

lowest     888868.992880
second     845517.726309
middle     769568.995726
fourth     678161.673963
highest    535001.677378
dtype: float64

In [20]:
s_births_and_stillbirths_by_wealth = (
    births_and_stillbirths
    * dist_births_and_stillbirths_by_wealth
    / dist_births_and_stillbirths_by_wealth.sum()
)
s_births_and_stillbirths_by_wealth

lowest     904203.941603
second     860104.770160
middle     782845.756624
fourth     689861.457147
highest    544231.635173
dtype: float64

In [21]:
# http://ihmeuw.org/6jr2 -- extracted from GBD Foresight, count of NTD deaths for under-1 year olds
if location == "india":
    ntd_deaths = 4_273.37
elif location == "nigeria":
    ntd_deaths = 5_373.52
elif location == "ethiopia":
    ntd_deaths = 1_883.76


ntd_death_rate = ntd_deaths / n_births
10_000 * ntd_death_rate

5.067795694522652

In [22]:
# Assumed does not vary by wealth
champs_ntd_stillbirth_per_livebirth = 51 / (69 - 51)
ntd_stillbirths = ntd_deaths * champs_ntd_stillbirth_per_livebirth
10_000 * (
    ntd_deaths + ntd_stillbirths
) / n_births  # ntd rate, compare with 41 per 10,000 from Bhide et al https://pubmed.ncbi.nlm.nih.gov/23873811/

19.426550162336838

We could not find a good source for folate intake by wealth in India, or even a representative source for overall folate intake.  [This paper](https://www.ncbi.nlm.nih.gov/pmc/articles/PMC10755415/pdf/S1368980023002112a.pdf) has an overall number of 220 mcg/day for women, and since it is not too different from the values we found in Ethiopia and Nigeria, we are going to use it for now. (It also has standard deviation of 50, which we can use when we introduce heterogeneity)

In [23]:
if location == "india":
    s_baseline_folate = pd.Series(
        dict(
            lowest=220,
            second=220,
            middle=220,
            fourth=220,
            highest=220,  # NRV is 400 mcg/day
        )
    )
elif location == "nigeria":
    # Table 95 of NFCMS 2021 Report
    s_baseline_folate = pd.Series(
        dict(
            lowest=189,
            second=198,
            middle=197,
            fourth=203,
            highest=208,  # NRV is 400 mcg/day
        )
    )  # how can we use this to estimate disparities in NTD?
elif location == "ethiopia":
    # Table 6 of https://cdn.nutrition.org/article/S2475-2991%2824%2901728-1/fulltext
    s_baseline_folate = pd.Series(
        dict(
            lowest=166,
            second=152,
            middle=137,
            fourth=350,
            highest=469,  # NRV is 400 mcg/day
        )
    )  # how can we use this to estimate disparities in NTD?

In [24]:
if location == "india":
    s_dist_deaths_by_wealth = pd.Series(  # Table 7.9 on page 201 of the CNNS report has RBC folate deficiency rates;
        dict(  # it includes wealth stratification, but has a very low threshold for insufficiency
            lowest=1,  # so I am assuming that most everyone is in the danger zone for low folate
            second=1,
            middle=1,
            fourth=1,
            highest=1,
        )
    )
elif location == "nigeria":
    s_dist_deaths_by_wealth = pd.Series(  # assume same rate for all, for now;
        dict(  # can CHAMPS offer more detail?  Need to infer wealth somehow
            lowest=1,
            second=1,
            middle=1,
            fourth=1,
            highest=1,
        )
    )
elif location == "ethiopia":
    s_dist_deaths_by_wealth = pd.Series(
        dict(  # supplementation studies don't make this easy, but here is a guess
            lowest=1,
            second=1,
            middle=1,
            fourth=1,
            highest=1,
        )
    )
    s_dist_deaths_by_wealth /= s_dist_deaths_by_wealth.mean()

s_dist_deaths_by_wealth

lowest     1.0
second     1.0
middle     1.0
fourth     1.0
highest    1.0
dtype: float64

In [25]:
s_ntd_death_rate = ntd_deaths / n_births * s_dist_deaths_by_wealth
10_000 * s_ntd_death_rate

lowest     5.067796
second     5.067796
middle     5.067796
fourth     5.067796
highest    5.067796
dtype: float64

In [26]:
s_ntd_death_count = s_ntd_death_rate * s_births_and_stillbirths_by_wealth
s_ntd_death_count

lowest     458.232084
second     435.883525
middle     396.730235
fourth     349.607692
highest    275.805474
dtype: float64

In [27]:
s_ntd_death_count.sum(), ntd_deaths  # should be similar

(1916.2590108076206, 1883.76)

In [28]:
s_ntd_stillbirth_count = s_ntd_death_count * champs_ntd_stillbirth_per_livebirth
s_ntd_stillbirth_count

lowest     1298.324239
second     1235.003321
middle     1124.069001
fourth      990.555128
highest     781.448842
dtype: float64

In [29]:
s_ntd_death_or_stillbirth_count = s_ntd_death_count + s_ntd_stillbirth_count
s_ntd_death_or_stillbirth_count

lowest     1756.556323
second     1670.886846
middle     1520.799236
fourth     1340.162820
highest    1057.254316
dtype: float64

In [30]:
def backcalc_rbc(ntd_risk, method):
    """
    ln (odds of NTD risk) = 1.6563 − 1.2193 × ln (RBC) (Daly et al, 1995)
    ln (odds of NTD risk) = 4.57 − 1.70 × ln (RBC) (Crider et al, 2014)
    """

    odds = ntd_risk / (1 - ntd_risk)
    ln_odds = np.log(odds)
    if method == "daly":
        neg_ln_rbc = (ln_odds - 1.6563) / 1.2193
    elif method == "crider":
        neg_ln_rbc = (ln_odds - 4.57) / 1.70
    rbc = np.exp(-neg_ln_rbc)
    return rbc


backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, "daly")

lowest     641.286398
second     641.286398
middle     641.286398
fourth     641.286398
highest    641.286398
dtype: float64

In [31]:
backcalc_rbc(
    s_ntd_death_or_stillbirth_count / s_births, "crider"
)  # compare with CNNS, https://www.unicef.org/india/media/2646/file/CNNS-report.pdf in Table 7.9

lowest     572.363568
second     572.363568
middle     572.363568
fourth     572.363568
highest    572.363568
dtype: float64

In [32]:
if location == "india":
    assert vehicle == "rice"
    s_daily_vehicle = pd.Series(  # Zeb and Alix analysis of HCES
        dict(
            lowest=213.570675,  # grams
            second=175.027768,
            middle=163.699804,
            fourth=163.363078,
            highest=127.874174,
        )
    )
elif location == "nigeria":
    assert vehicle == "bouillon"
    s_daily_vehicle = pd.Series(  # NFCMS 2021, Table 170. Usual intake of Bouillon (raw weight, grams) of women
        dict(
            lowest=8.4,  # grams
            second=8.0,
            middle=5.9,
            fourth=4.9,
            highest=4.6,
        )
    )
elif location == "ethiopia":
    assert vehicle == "salt"
    s_daily_vehicle = (
        pd.Series(  # Dememoz Woldegebreal, personal communication of analysis
            dict(  # of 2013 Ethiopian National Food Consumption Survey (ENFCS)
                lowest=7.5467,  # grams
                second=6.3605,
                middle=6.5508,
                fourth=6.5491,
                highest=6.5406,
            )
        )
        * 0.90
    )  # Saje et al 2024 assume 90% of total salt consumption comes from discretionary salt and manufactured food items (cites James et al 1987 )

In [33]:
baseline_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/concentration/{location}.csv"
)
assert (baseline_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert baseline_concentration_mcg_per_gram.value.nunique() == 1
baseline_concentration_mcg_per_gram = baseline_concentration_mcg_per_gram.value.iloc[0]
baseline_concentration_mcg_per_gram

0.0

In [34]:
intervention_concentration_mcg_per_gram = pd.read_csv(
    f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/concentration/{location}.csv"
)
assert (intervention_concentration_mcg_per_gram.vehicle_name == vehicle).all()
assert intervention_concentration_mcg_per_gram.value.nunique() == 1
intervention_concentration_mcg_per_gram = (
    intervention_concentration_mcg_per_gram.value.iloc[0]
)
intervention_concentration_mcg_per_gram

14.084507042253522

In [35]:
eff_fort_baseline_path = f"../0100_data_prep/results/folate/{vehicle}/baseline_fortification/effective_coverage/{location}.csv"
eff_fort_intervention_path = f"../0100_data_prep/results/folate/{vehicle}/{intervention_scenario}/intervention_fortification/effective_coverage/{location}.csv"

df_eff_fort_baseline = pd.read_csv(eff_fort_baseline_path)
assert (df_eff_fort_baseline.vehicle_name == vehicle).all()
df_eff_fort_intervention = pd.read_csv(eff_fort_intervention_path)
assert (df_eff_fort_intervention.vehicle_name == vehicle).all()

In [36]:
# NOTE: Using DHS definition of WRA
population = (
    pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
    .groupby(["sex", "age_start", "age_end", "wealth_quintile"])
    .value.sum()
    .reset_index()
)
population = population[
    (population.sex == "Female")
    & (population.age_start >= 15)
    & (population.age_end <= 50)
]
population

,sex,age_start,age_end,wealth_quintile,value
40,Female,15.0,20.0,fourth,1.212617e+06
41,Female,15.0,20.0,highest,1.851710e+06
42,Female,15.0,20.0,lowest,9.914727e+05
43,Female,15.0,20.0,middle,1.134915e+06
...,...,...,...,...,...
71,Female,45.0,50.0,highest,4.350470e+05
72,Female,45.0,50.0,lowest,3.256086e+05
73,Female,45.0,50.0,middle,3.622846e+05
74,Female,45.0,50.0,second,3.254899e+05


In [37]:
if "sex" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[(df_eff_fort_baseline.sex == "Female")]

if "age_start" in df_eff_fort_baseline.columns:
    df_eff_fort_baseline = df_eff_fort_baseline[
        (df_eff_fort_baseline.age_start >= 15) & (df_eff_fort_baseline.age_end <= 50)
    ]

In [38]:
if "sex" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.sex == "Female")
    ]

if "age_start" in df_eff_fort_intervention.columns:
    df_eff_fort_intervention = df_eff_fort_intervention[
        (df_eff_fort_intervention.age_start >= 15)
        & (df_eff_fort_intervention.age_end <= 50)
    ]

In [39]:
def aggregate_using_population(effective_fort):
    merge_cols = [
        c
        for c in ["sex", "wealth_quintile", "age_start", "age_end"]
        if c in effective_fort.columns
    ]
    merged = effective_fort.merge(
        population.reset_index(),
        on=[c for c in merge_cols if "age" not in c],
        suffixes=("_fort", "_pop"),
    )
    assert ("age_start" in merge_cols) == ("age_end" in merge_cols)
    if "age_start" in merge_cols:
        merged = merged[
            (merged.age_start_pop >= merged.age_start_fort)
            & (merged.age_end_pop <= merged.age_end_fort)
        ]
    print(merged)
    assert len(merged) == len(population)
    return merged.groupby(["wealth_quintile"]).apply(
        lambda df: (df.value_fort * df.value_pop).sum() / df.value_pop.sum()
    )

In [40]:
df_eff_fort_baseline = aggregate_using_population(df_eff_fort_baseline)
df_eff_fort_baseline

   vehicle_name wealth_quintile  value_fort  index     sex  age_start  \
0          salt          fourth         0.0     40  Female       15.0   
1          salt          fourth         0.0     45  Female       20.0   
2          salt          fourth         0.0     50  Female       25.0   
3          salt          fourth         0.0     55  Female       30.0   
..          ...             ...         ...    ...     ...        ...   
31         salt          second         0.0     59  Female       30.0   
32         salt          second         0.0     64  Female       35.0   
33         salt          second         0.0     69  Female       40.0   
34         salt          second         0.0     74  Female       45.0   

    age_end     value_pop  
0      20.0  1.212617e+06  
1      25.0  1.034256e+06  
2      30.0  8.686004e+05  
3      35.0  7.670239e+05  
..      ...           ...  
31     35.0  6.784588e+05  
32     40.0  5.671793e+05  
33     45.0  4.535687e+05  
34     50.0  3.25

wealth_quintile
fourth     0.0
highest    0.0
lowest     0.0
middle     0.0
second     0.0
dtype: float64

In [41]:
df_eff_fort_intervention = aggregate_using_population(df_eff_fort_intervention)
df_eff_fort_intervention

   vehicle_name wealth_quintile     sex  value_fort  index  age_start  \
0          salt          fourth  Female        0.64     40       15.0   
1          salt          fourth  Female        0.64     45       20.0   
2          salt          fourth  Female        0.64     50       25.0   
3          salt          fourth  Female        0.64     55       30.0   
..          ...             ...     ...         ...    ...        ...   
31         salt          second  Female        0.64     59       30.0   
32         salt          second  Female        0.64     64       35.0   
33         salt          second  Female        0.64     69       40.0   
34         salt          second  Female        0.64     74       45.0   

    age_end     value_pop  
0      20.0  1.212617e+06  
1      25.0  1.034256e+06  
2      30.0  8.686004e+05  
3      35.0  7.670239e+05  
..      ...           ...  
31     35.0  6.784588e+05  
32     40.0  5.671793e+05  
33     45.0  4.535687e+05  
34     50.0  3.25

wealth_quintile
fourth     0.64
highest    0.64
lowest     0.64
middle     0.64
second     0.64
dtype: float64

In [42]:
quintile_name_map = {
    "lowest": "lowest",
    "second": "second",
    "middle": "middle",
    "fourth": "fourth",
    "highest": "highest",
}
df_eff_fort_baseline.index = df_eff_fort_baseline.index.map(quintile_name_map)
df_eff_fort_intervention.index = df_eff_fort_intervention.index.map(quintile_name_map)

In [43]:
RBC_baseline = backcalc_rbc(s_ntd_death_or_stillbirth_count / s_births, "crider")
RBC_baseline

lowest     572.363568
second     572.363568
middle     572.363568
fourth     572.363568
highest    572.363568
dtype: float64

In [44]:
# Fortification folate needs to be converted into dietary folate equivalents (DFEs)
# for use with our effect size.
# https://www.jandonline.org/article/S0002-8223(00)00027-4/pdf
fortification_mcg_to_dfe = 1.7

In [45]:
s_intervention_folate = (
    s_baseline_folate
    - (
        df_eff_fort_baseline
        * s_daily_vehicle
        * baseline_concentration_mcg_per_gram
        * fortification_mcg_to_dfe
    )
    + (
        df_eff_fort_intervention
        * s_daily_vehicle
        * intervention_concentration_mcg_per_gram
        * fortification_mcg_to_dfe
    )
)
s_intervention_folate

fourth     440.322235
highest    559.205007
lowest     270.080685
middle     227.345681
second     239.721149
dtype: float64

In [46]:
intevention_folate_pct_increase = (
    s_intervention_folate - s_baseline_folate
) / s_baseline_folate
intevention_folate_pct_increase

fourth     0.258064
highest    0.192335
lowest     0.626992
middle     0.659458
second     0.577113
dtype: float64

In [47]:
RBC_with_fort = RBC_baseline * (1 + ((6 / 10) * intevention_folate_pct_increase))
RBC_with_fort

fourth     660.987266
highest    638.414818
lowest     787.684022
middle     798.833246
second     770.554582
dtype: float64

In [48]:
def calc_ntd_pr(df, method):
    ln_rbc = np.log(df)
    if method == "daly":
        ln_odds = 1.6563 - 1.2193 * ln_rbc
    elif method == "crider":
        ln_odds = 4.57 - 1.70 * ln_rbc
    p = np.exp(ln_odds)  # TODO: better transformation
    return p


s_ntd_death_or_stillbirth_rate_with_fort = calc_ntd_pr(RBC_with_fort, "crider")
10_000 * s_ntd_death_or_stillbirth_rate_with_fort

fourth     15.502350
highest    16.445639
lowest     11.506078
middle     11.234412
second     11.944280
dtype: float64

In [49]:
s_ntd_death_or_stillbirth_count_with_fort = (
    s_ntd_death_or_stillbirth_rate_with_fort * s_births
)
s_ntd_death_or_stillbirth_count_with_fort

fourth     1051.309930
highest     879.844445
lowest     1022.739613
middle      864.565541
second     1009.910081
dtype: float64

In [50]:
ntd_cases_by_scenario = pd.concat(
    [
        s_ntd_death_or_stillbirth_count.rename("value")
        .rename_axis("wealth_quintile")
        .to_frame()
        .assign(entity="ntd", scenario="baseline")
        .set_index(["entity", "scenario"], append=True)
        .value,
        s_ntd_death_or_stillbirth_count_with_fort.rename("value")
        .rename_axis("wealth_quintile")
        .to_frame()
        .assign(entity="ntd", scenario=intervention_scenario)
        .set_index(["entity", "scenario"], append=True)
        .value,
    ]
)
ntd_cases_by_scenario

wealth_quintile  entity  scenario           
lowest           ntd     baseline               1756.556323
second           ntd     baseline               1670.886846
middle           ntd     baseline               1520.799236
fourth           ntd     baseline               1340.162820
                                                   ...     
highest          ntd     intervention_25_nrv     879.844445
lowest           ntd     intervention_25_nrv    1022.739613
middle           ntd     intervention_25_nrv     864.565541
second           ntd     intervention_25_nrv    1009.910081
Name: value, Length: 10, dtype: float64

In [51]:
path = (
    f"./results/{location}/{vehicle}/{intervention_scenario}/ntd_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)

In [52]:
# For calculating YLLs
tmrle = vivarium_inputs.get_theoretical_minimum_risk_life_expectancy()
tmrle

,,value
age_start,age_end,
0.00,0.01,89.958040
0.01,0.02,89.975474
0.02,0.03,89.990990
0.03,0.04,89.985077
...,...,...
109.97,109.98,4.509941
109.98,109.99,4.504631
109.99,110.00,4.499321
110.00,125.00,4.494011


In [53]:
# NOTE: Treating stillbirths as a death!
yll_per_ntd = float(tmrle.iloc[0])
yll_per_ntd

89.95803974533831

In [54]:
ylls_by_scenario = ntd_cases_by_scenario * yll_per_ntd
ylls_by_scenario

wealth_quintile  entity  scenario           
lowest           ntd     baseline               158016.363506
second           ntd     baseline               150309.705324
middle           ntd     baseline               136808.118121
fourth           ntd     baseline               120558.420248
                                                    ...      
highest          ntd     intervention_25_nrv     79149.081598
lowest           ntd     intervention_25_nrv     92003.650742
middle           ntd     intervention_25_nrv     77774.621310
second           ntd     intervention_25_nrv     90849.531217
Name: value, Length: 10, dtype: float64

In [55]:
path = f"./results/{location}/{vehicle}/{intervention_scenario}/ylls_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ylls_by_scenario.to_csv(path)